In [1]:
# ===== Phase 5, Batch A: log-path check + determinism check (wf-Aug) =====
import sys, hashlib, importlib
from pathlib import Path
from functools import partial
sys.path.insert(0, str(Path.cwd().parent / "src"))          # notebook lives in notebooks/

import numpy as np, pandas as pd, sklearn
import config, features, baselines, validation, models, experiments
for m in (config, features, baselines, validation, models, experiments):
    importlib.reload(m)                                       # pick up today's edits
from config import TRAIN_PATH, OUTPUT_DIR, PHASE5_LOG_PATH
from sklearn.ensemble import HistGradientBoostingRegressor
print("sklearn", sklearn.__version__)

# --- A0. Fingerprint the Phase 4 log so we can prove Phase 5 never touches it ---
P4 = OUTPUT_DIR / "phase4_experiments.csv"
md5 = lambda p: hashlib.md5(p.read_bytes()).hexdigest()
p4_before = md5(P4)

# --- A1. What do the 1.9.1 docs say random_state controls? (read before relying on it) ---
doc = HistGradientBoostingRegressor.__doc__
i = doc.index("random_state :")
print("\n--- docs: random_state ---\n" + doc[i:i + 700])

# --- A2. Factories: FINAL (D42) and the same model with random_state=1 ---
FINAL = partial(models.HGBRate, target="log_rpm", extra=("weekday",),
                drop=("market_index",), learning_rate=0.05, max_iter=400)
SEED1 = partial(FINAL, random_state=1)        # overrides the SEED default inside HGBRate

built = {}                                     # keep each fitted model so we can read its real settings
def capture(factory, key):
    def make():
        built[key] = factory()
        return built[key]
    return make

raw = pd.read_csv(TRAIN_PATH)
AUG = validation.WALK_FOLDS[-1]
assert AUG.name == "wf-Aug"

# Same seed twice (repeatability), then seed 1 (does the seed matter?)
runs = {key: validation.run_fold(raw, AUG, capture(fac, key))
        for key, fac in [("seed42", FINAL), ("seed42_again", FINAL), ("seed1", SEED1)]}

# --- A3. Real estimator settings (not assumed) ---
print("\n--- fitted estimator settings ---")
for key, m in built.items():
    p = m.model_.get_params()
    print(f"{key:13s}| {m.name}\n{'':13s}| " +
          ", ".join(f"{k}={p.get(k, 'n/a')}" for k in
                    ["learning_rate", "max_iter", "early_stopping", "random_state", "max_features"]))

# --- A4. Prediction checks + determinism ---
base_idx = runs["seed42"]["holdout"].index
assert all(r["holdout"].index.equals(base_idx) for r in runs.values()), "row order differs"
preds = {k: r["holdout"]["pred"].to_numpy() for k, r in runs.items()}
print("\n--- predictions ---")
for k, p in preds.items():
    print(f"{k:13s} n={len(p)} (holdout {runs[k]['test_rows']}) finite={np.isfinite(p).all()} "
          f">0={(p > 0).all()} MAE={runs[k]['scores']['MAE']:.4f}")
d_same = np.abs(preds["seed42"] - preds["seed42_again"]).max()
d_seed = np.abs(preds["seed42"] - preds["seed1"]).max()
print(f"\nmax |seed42 - seed42 again| = {d_same:.6g}")
print(f"max |seed42 - seed1|        = {d_seed:.6g}")
print("DETERMINISTIC (seed changes nothing):", d_seed == 0)

# --- A5. Reproduce the logged Phase 4 MAE for FINAL on wf-Aug (must be 89.99) ---
p4 = pd.read_csv(P4)
ref = p4[(p4["fold"] == "wf-Aug") & (p4["model"] == built["seed42"].name)
         & (p4["drop_errors"].astype(str) == "True")]
assert len(ref) == 1, f"expected 1 logged FINAL wf-Aug row, found {len(ref)}"
print(f"\nlogged MAE {ref['MAE'].item():.4f} vs now {runs['seed42']['scores']['MAE']:.4f}")
assert abs(ref["MAE"].item() - runs["seed42"]["scores"]["MAE"]) < 1e-3, "FINAL does not reproduce: STOP"

# --- A6. Log to the Phase 5 file, reusing Phase 4's label strings for consistency ---
tgt, feats, sett = ref[["target", "features", "settings"]].iloc[0]
experiments.log_experiments(
    [experiments.experiment_row(runs["seed42"], tgt, feats, sett),
     experiments.experiment_row(runs["seed1"], tgt, feats, f"{sett} random_state=1")],
    path=PHASE5_LOG_PATH)
print(f"\nPhase 5 log rows: {len(pd.read_csv(PHASE5_LOG_PATH))} (expect 2)")
print(f"Phase 4 log rows: {len(pd.read_csv(P4))} (expect 63), unchanged: {md5(P4) == p4_before}")

sklearn 1.9.1

--- docs: random_state ---
random_state : int, RandomState instance or None, default=None
        Pseudo-random number generator to control the subsampling in the
        binning process, and the train/validation data split if early stopping
        is enabled.
        Pass an int for reproducible output across multiple function calls.
        See :term:`Glossary <random_state>`.

    Attributes
    ----------
    do_early_stopping_ : bool
        Indicates whether early stopping is used during training.
    n_iter_ : int
        The number of iterations as selected by early stopping, depending on
        the `early_stopping` parameter. Otherwise it corresponds to max_iter.
    n_trees_per_iteration_ : int
        The

--- fitted estimator settings ---
seed42       | HGB log_rpm -market_index +weekday learning_rate=0.05 max_iter=400
             | learning_rate=0.05, max_iter=400, early_stopping=False, random_state=42, max_features=1.0
seed42_again | HGB log_rpm -market_

In [2]:
# ===== Phase 5, Batch B: FINAL + 3 variants on the 4 walk folds; FINAL on Sep-Oct (report only) =====
from config import PROJECT_ROOT

WALK = validation.WALK_FOLDS
VARIANTS = {   # name: (factory, drop_errors)
    "FINAL":            (FINAL, True),
    "NO-WEEKDAY":       (partial(models.HGBRate, target="log_rpm", extra=(), drop=("market_index",),
                                 learning_rate=0.05, max_iter=400), True),
    "DEFAULT SETTINGS": (partial(models.HGBRate, target="log_rpm", extra=("weekday",),
                                 drop=("market_index",)), True),
    "ERRORS KEPT":      (FINAL, False),
}

# --- B0. Pre-run checks: real estimator settings for each variant, BEFORE training ---
print("--- pre-run settings ---")
params = {}
for v, (fac, de) in VARIANTS.items():
    m = fac()
    params[v] = HistGradientBoostingRegressor(**m.settings).get_params()
    p = params[v]
    print(f"{v:17s}| {m.name} | drop_errors={de}\n{'':17s}| " +
          ", ".join(f"{k}={p[k]}" for k in ["learning_rate", "max_iter", "early_stopping", "random_state"]))
others = lambda p: {k: val for k, val in p.items() if k not in ("learning_rate", "max_iter")}
assert params["DEFAULT SETTINGS"]["early_stopping"] is False and params["DEFAULT SETTINGS"]["random_state"] == 42
assert others(params["DEFAULT SETTINGS"]) == others(params["FINAL"]), "DEFAULT differs beyond lr/max_iter"
assert (params["DEFAULT SETTINGS"]["learning_rate"], params["DEFAULT SETTINGS"]["max_iter"]) != (0.05, 400)
assert params["NO-WEEKDAY"] == params["FINAL"] == params["ERRORS KEPT"], "only features/drop_errors may differ"
print("pre-run checks passed")

def logged(model_name, fold, drop_errors):
    """Phase 4 log row for this run, or None."""
    hit = p4[(p4["model"] == model_name) & (p4["fold"] == fold)
             & (p4["drop_errors"].astype(str) == str(drop_errors))]
    return hit.iloc[0] if len(hit) == 1 else None

def check(r):
    p = r["holdout"]["pred"].to_numpy()
    ok = len(p) == r["test_rows"] and np.isfinite(p).all() and (p > 0).all()
    print(f"  {r['fold']:7s} rows={r['test_rows']:5d} ok={ok} dropped={r['dropped']:4d} "
          f"fit_rows={r['fit_rows']:6d}/{r['cleaning_rows']:6d} MAE={r['scores']['MAE']:.4f}")
    assert ok

# --- B1. FINAL first: must reproduce its logged per-fold MAEs before anything new runs ---
res = {}
print("\n--- FINAL (reproduction gate) ---")
res["FINAL"] = [validation.run_fold(raw, f, FINAL, True) for f in WALK]
for r in res["FINAL"]:
    check(r)
    ref_row = logged(r["model"], r["fold"], True)
    assert ref_row is not None and abs(ref_row["MAE"] - r["scores"]["MAE"]) < 1e-3, f"FINAL {r['fold']} does not reproduce: STOP"
print("FINAL reproduces May/Jun/Jul/Aug:", [round(r["scores"]["MAE"], 2) for r in res["FINAL"]])

# --- B2. The three variants ---
for v in ["NO-WEEKDAY", "DEFAULT SETTINGS", "ERRORS KEPT"]:
    fac, de = VARIANTS[v]
    print(f"\n--- {v} ---")
    res[v] = [validation.run_fold(raw, f, fac, de) for f in WALK]
    for r in res[v]:
        check(r)
        prev = logged(r["model"], r["fold"], de)            # cross-check if Phase 4 ran the same thing
        if prev is not None:
            print(f"    Phase 4 logged {prev['MAE']:.4f}, match={abs(prev['MAE'] - r['scores']['MAE']) < 1e-3}")
# drop_errors really reached run_fold:
assert all(r["drop_errors"] is False and r["dropped"] == 0 and r["fit_rows"] == r["cleaning_rows"] for r in res["ERRORS KEPT"])
assert all(r["drop_errors"] is True and r["dropped"] > 0 for r in res["FINAL"])
print("\ndrop_errors check passed (ERRORS KEPT trains on every row; FINAL drops flagged rows)")

# --- B3. FINAL on Sep-Oct: REPORT ONLY (deterministic, so this reproduces the second look; nothing is chosen from it) ---
main_final = validation.run_fold(raw, validation.MAIN_FOLD, FINAL, True)
print("\n--- FINAL on Sep-Oct (report only) ---"); check(main_final)
mref = logged(main_final["model"], "main", True)
print(f"  Phase 4 logged {mref['MAE']:.4f} -> reproduces: {abs(mref['MAE'] - main_final['scores']['MAE']) < 1e-3}")

# --- B4. Log every run to the Phase 5 file (reuse Phase 4 label strings where the run exists there) ---
fallback = {"FINAL": (tgt, feats, sett), "ERRORS KEPT": (tgt, feats, sett),
            "NO-WEEKDAY": (tgt, f"{feats} MINUS weekday", sett),
            "DEFAULT SETTINGS": (tgt, feats, "defaults (lr/max_iter not set)")}
rows, label_src = [], {}
for v, rs in res.items():
    for r in rs:
        prev = logged(r["model"], r["fold"], r["drop_errors"])
        lab = tuple(prev[["target", "features", "settings"]]) if prev is not None else fallback[v]
        label_src[v] = "reused from Phase 4" if prev is not None else "new"
        rows.append(experiments.experiment_row(r, *lab))
rows.append(experiments.experiment_row(main_final, tgt, feats, sett))
experiments.log_experiments(rows, path=PHASE5_LOG_PATH)
print("\nlabel strings:", label_src)
print(f"Phase 5 log rows: {len(pd.read_csv(PHASE5_LOG_PATH))} (expect 18 = seed1 + 16 walk + 1 Sep-Oct)")
print(f"Phase 4 log unchanged: {md5(P4) == p4_before}")

# --- B5. Keep holdout predictions for the bootstrap: memory + scratch/ (git-ignored), never outputs/ ---
frames = []
for v, rs in list(res.items()) + [("FINAL", [main_final])]:
    for r in rs:
        h = r["holdout"]
        frames.append(pd.DataFrame({"variant": v, "fold": r["fold"], "load_id": h["load_id"].to_numpy(),
                                    "date": h["date"].to_numpy(), "posted_rate": h["posted_rate"].to_numpy(),
                                    "pred": h["pred"].to_numpy(), "is_label_error": h["is_label_error"].to_numpy()}))
preds_long = pd.concat(frames, ignore_index=True)
for f in WALK:   # paired bootstrap needs identical rows in identical order for every variant
    ids = [preds_long.query("variant == @v and fold == @f.name")["load_id"].tolist() for v in res]
    assert all(i == ids[0] for i in ids), f"{f.name}: variants scored different rows"
SCRATCH = PROJECT_ROOT / "scratch"; SCRATCH.mkdir(exist_ok=True)
preds_long.to_csv(SCRATCH / "phase5_holdout_preds.csv", index=False)
print(f"\nsaved {len(preds_long):,} holdout predictions to scratch/ (pairing check passed)")

# --- B6. Raw numbers only (judgement is Batch C) ---
summ = experiments.walk_summary(res, ref="FINAL")
cols = [c for c in summ.columns if c.startswith("MAE ")] + ["mean_MAE", "mean_MAPE_%", "mean_clean_MAE",
        "dMAE_vs_ref", "dMAPE_pp_vs_ref", "wins_vs_ref"]
print("\n(wins_vs_ref = folds where the VARIANT beats FINAL)")
print(summ[cols].round(2).to_string())

--- pre-run settings ---
FINAL            | HGB log_rpm -market_index +weekday learning_rate=0.05 max_iter=400 | drop_errors=True
                 | learning_rate=0.05, max_iter=400, early_stopping=False, random_state=42
NO-WEEKDAY       | HGB log_rpm -market_index learning_rate=0.05 max_iter=400 | drop_errors=True
                 | learning_rate=0.05, max_iter=400, early_stopping=False, random_state=42
DEFAULT SETTINGS | HGB log_rpm -market_index +weekday | drop_errors=True
                 | learning_rate=0.1, max_iter=100, early_stopping=False, random_state=42
ERRORS KEPT      | HGB log_rpm -market_index +weekday learning_rate=0.05 max_iter=400 | drop_errors=False
                 | learning_rate=0.05, max_iter=400, early_stopping=False, random_state=42
pre-run checks passed

--- FINAL (reproduction gate) ---
  wf-May  rows= 4913 ok=True dropped= 275 fit_rows= 18835/ 19110 MAE=155.8138
  wf-Jun  rows= 4783 ok=True dropped= 337 fit_rows= 23686/ 24023 MAE=212.1455
  wf-Jul  rows= 491

In [3]:
# ===== Phase 5, Batch C: D36 by code -> re-runs on submitted model -> third look -> paired day-block bootstrap =====
from config import SEED, PHASE5_BOOTSTRAP_PATH

# --- C1. D36 reversal rule, pre-registered order: D39 first, judged on the tuned-model folds ---
# Reverse only if the SIMPLER option is <= $1 worse (mean MAE) AND mean MAPE rises <= 0.10 pp.
simpler_ok = lambda row: bool(row["dMAE_vs_ref"] <= 1.0 and row["dMAPE_pp_vs_ref"] <= 0.10)
r39 = summ.loc["DEFAULT SETTINGS"]
reverse_D39 = simpler_ok(r39)
print(f"D39 re-check: defaults vs tuned  dMAE={r39['dMAE_vs_ref']:+.2f}  dMAPE={r39['dMAPE_pp_vs_ref']:+.2f} pp  "
      f"-> {'REVERSE: use library defaults' if reverse_D39 else 'KEEP tuned settings'}")
DEFAULTS = VARIANTS["DEFAULT SETTINGS"][0]
NEW = DEFAULTS if reverse_D39 else FINAL
tuned_kw = {} if reverse_D39 else {"learning_rate": 0.05, "max_iter": 400}
NO_WD_NEW = partial(models.HGBRate, target="log_rpm", extra=(), drop=("market_index",), **tuned_kw)
print("SUBMITTED model:", NEW().name)

# --- C2. Re-run the two comparisons on the SUBMITTED model's settings (decision a) ---
DEF_LABEL = "defaults (lr/max_iter not set)"
if reverse_D39:
    pn, pw = (HistGradientBoostingRegressor(**fac().settings).get_params() for fac in (NO_WD_NEW, NEW))
    assert pn == pw, "NO-WEEKDAY (new) must use exactly the submitted settings"
    print(f"pre-run check: lr={pw['learning_rate']}, max_iter={pw['max_iter']}, "
          f"early_stopping={pw['early_stopping']}, random_state={pw['random_state']}")
    res_new = {"FINAL": res["DEFAULT SETTINGS"]}          # already run in Batch B (deterministic)
    for v, fac, de in [("NO-WEEKDAY", NO_WD_NEW, True), ("ERRORS KEPT", NEW, False)]:
        print(f"\n--- {v} (submitted settings) ---")
        res_new[v] = [validation.run_fold(raw, f, fac, de) for f in WALK]
        for r in res_new[v]:
            check(r)
            prev = logged(r["model"], r["fold"], de)
            if prev is not None:
                print(f"    Phase 4 logged {prev['MAE']:.4f}, match={abs(prev['MAE'] - r['scores']['MAE']) < 1e-3}")
    assert all(r["dropped"] == 0 and r["fit_rows"] == r["cleaning_rows"] for r in res_new["ERRORS KEPT"])
    fb = {"NO-WEEKDAY": (tgt, f"{feats} MINUS weekday", DEF_LABEL), "ERRORS KEPT": (tgt, feats, DEF_LABEL)}
    rows = []
    for v in ["NO-WEEKDAY", "ERRORS KEPT"]:
        for r in res_new[v]:
            prev = logged(r["model"], r["fold"], r["drop_errors"])
            rows.append(experiments.experiment_row(r, *(tuple(prev[["target", "features", "settings"]])
                                                        if prev is not None else fb[v])))
    experiments.log_experiments(rows, path=PHASE5_LOG_PATH)
else:
    res_new = {k: res[k] for k in ["FINAL", "NO-WEEKDAY", "ERRORS KEPT"]}

summ_new = experiments.walk_summary(res_new, ref="FINAL")       # "FINAL" here = SUBMITTED model
print("\n(ref = SUBMITTED; wins_vs_ref = folds where the VARIANT beats it)")
print(summ_new[cols].round(2).to_string())

r38 = summ_new.loc["ERRORS KEPT"]
reverse_D38 = simpler_ok(r38)
print(f"\nD38 re-check on SUBMITTED: errors kept  dMAE={r38['dMAE_vs_ref']:+.2f}  dMAPE={r38['dMAPE_pp_vs_ref']:+.2f} pp  "
      f"-> {'REVERSE' if reverse_D38 else 'KEEP excluding label errors'}")
assert not reverse_D38, "D38 reverses on the submitted model: STOP and paste this output"
rw = summ_new.loc["NO-WEEKDAY"]
print(f"weekday (label only, stays in model): no-weekday dMAE={rw['dMAE_vs_ref']:+.2f}, "
      f"dMAPE={rw['dMAPE_pp_vs_ref']:+.2f} pp, SUBMITTED wins {4 - int(rw['wins_vs_ref'])}/4")

# --- C3. Third look at Sep-Oct: ONLY because a choice was reversed. REPORT ONLY; result is final either way ---
def sepoct(r):
    h = r["holdout"]; c = h[~h["is_label_error"]]
    return {"rows": r["test_rows"], "MAE": r["scores"]["MAE"], "MAPE_%": r["scores"]["MAPE_%"],
            "clean_MAE": validation.mae(c["posted_rate"], c["pred"]),
            "clean_bias_%": 100 * ((c["pred"] - c["posted_rate"]) / c["posted_rate"]).mean()}
if reverse_D39 or reverse_D38:
    third = validation.run_fold(raw, validation.MAIN_FOLD, NEW, True)
    print("\n--- Sep-Oct THIRD LOOK (report only, choice made from folds) ---"); check(third)
    print(pd.DataFrame({"SUBMITTED (third look)": sepoct(third),
                        "tuned (second look, D42)": sepoct(main_final)}).T.round(2).to_string())
    b1 = logged("B1 median rpm by equipment x band", "main", True)
    print("B1 Sep-Oct:", f"MAE {b1['MAE']:.2f}, MAPE {b1['MAPE_%']:.2f}%, clean {b1['clean_MAE']:.2f}"
          if b1 is not None else "not in Phase 4 log; D24: MAE 140.53, MAPE 6.40%")
    experiments.log_experiments([experiments.experiment_row(
        third, tgt, feats, f"{DEF_LABEL} | third look, report only, choice made from folds")], path=PHASE5_LOG_PATH)
print(f"\nPhase 5 log rows: {len(pd.read_csv(PHASE5_LOG_PATH))} (expect 27 = 18 + 8 re-runs + 1 third look)")
print(f"Phase 4 log unchanged: {md5(P4) == p4_before}")

# --- C4. Paired bootstrap: SUBMITTED vs each alternative, same resampled days for every model ---
B = 1000
COMP = {"SUBMITTED": res_new["FINAL"], "NO-WEEKDAY": res_new["NO-WEEKDAY"],
        "TUNED lr0.05x400": res["FINAL"], "ERRORS KEPT": res_new["ERRORS KEPT"]}
ALTS = [k for k in COMP if k != "SUBMITTED"]

parts = []                                    # one row per holdout load: absolute error of every model
for i, f in enumerate(WALK):
    h0 = COMP["SUBMITTED"][i]["holdout"]
    y = h0["posted_rate"].to_numpy()
    d = pd.DataFrame({"fold": f.name, "date": h0["date"].to_numpy()})
    for v, rs in COMP.items():
        h = rs[i]["holdout"]
        assert (h["load_id"].to_numpy() == h0["load_id"].to_numpy()).all(), f"{v} {f.name}: rows differ (not paired)"
        d[v] = np.abs(h["pred"].to_numpy() - y)
    parts.append(d)
E = pd.concat(parts, ignore_index=True)

def bootstrap(Es, unit):
    """Each repeat draws ONE set of days (or rows) with replacement and scores EVERY model on it (paired).
    Day unit: MAE = (sum of abs errors on drawn days) / (loads on drawn days), so a day's loads move together."""
    if unit == "day":
        g = Es.groupby(["fold", "date"])
        sums, cnt = g[list(COMP)].sum().to_numpy(), g.size().to_numpy().astype(float)
    else:
        sums, cnt = Es[list(COMP)].to_numpy(), np.ones(len(Es))
    rng = np.random.default_rng(SEED)         # fixed seed per scope -> each scope reproducible on its own
    n, out = len(cnt), np.empty((B, len(COMP)))
    for b in range(B):
        idx = rng.integers(0, n, n)
        out[b] = sums[idx].sum(axis=0) / cnt[idx].sum()
    return pd.DataFrame(out, columns=list(COMP)), n

scopes = [("pooled 4 folds", E, "day"), ("pooled 4 folds", E, "row")] + \
         [(f.name, E[E["fold"] == f.name], "day") for f in WALK]
brows = []
for scope, Es, unit in scopes:
    bs, n = bootstrap(Es, unit)
    obs = Es[list(COMP)].mean()               # observed MAE of each model on exactly these rows
    lo, hi = np.percentile(bs["SUBMITTED"], [2.5, 97.5])
    brows.append({"scope": scope, "unit": unit, "comparison": "SUBMITTED MAE", "n_units": n, "repeats": B,
                  "seed": SEED, "point": obs["SUBMITTED"], "boot_mean": bs["SUBMITTED"].mean(),
                  "ci_low": lo, "ci_high": hi, "excludes_0": np.nan})
    for a in ALTS:
        gap = bs[a] - bs["SUBMITTED"]          # positive = SUBMITTED is better
        lo, hi = np.percentile(gap, [2.5, 97.5])
        brows.append({"scope": scope, "unit": unit, "comparison": f"{a} minus SUBMITTED", "n_units": n,
                      "repeats": B, "seed": SEED, "point": obs[a] - obs["SUBMITTED"], "boot_mean": gap.mean(),
                      "ci_low": lo, "ci_high": hi, "excludes_0": bool(lo > 0 or hi < 0)})
boot = pd.DataFrame(brows)

# Sanity: per-fold observed gaps must equal the logged per-fold MAE differences
for i, f in enumerate(WALK):
    for a in ALTS:
        pt = boot.query("scope == @f.name and comparison == @a + ' minus SUBMITTED'")["point"].item()
        assert abs(pt - (COMP[a][i]["scores"]["MAE"] - COMP["SUBMITTED"][i]["scores"]["MAE"])) < 1e-6
print("\nbootstrap sanity: per-fold point gaps = MAE differences from the runs")

# --- C5. Noise rule (decision 3c): robust = pooled DAY-block 95% CI excludes 0 AND same direction in >= 3/4 folds ---
lab = []
for a in ALTS:
    fold_gaps = np.array([COMP[a][i]["scores"]["MAE"] - COMP["SUBMITTED"][i]["scores"]["MAE"] for i in range(4)])
    pooled = boot.query("scope == 'pooled 4 folds' and unit == 'day' and comparison == @a + ' minus SUBMITTED'").iloc[0]
    direction = np.sign(pooled["point"])
    same = int((np.sign(fold_gaps) == direction).sum())
    label = "robust" if (pooled["excludes_0"] and same >= 3) else "within noise"
    boot.loc[pooled.name, ["folds_same_direction", "label"]] = [same, label]
    lab.append({"comparison": a, "better": "SUBMITTED" if direction > 0 else a,
                "fold-mean gap (D36)": fold_gaps.mean(), "pooled gap": pooled["point"],
                "day CI": f"[{pooled['ci_low']:+.2f}, {pooled['ci_high']:+.2f}]",
                "fold gaps May/Jun/Jul/Aug": " / ".join(f"{g:+.2f}" for g in fold_gaps),
                "same dir": f"{same}/4", "label": label})
boot.round(4).to_csv(PHASE5_BOOTSTRAP_PATH, index=False)

print("\n--- bootstrap table (gap = alternative minus SUBMITTED; + means SUBMITTED better) ---")
print(boot.drop(columns=["repeats", "seed"]).round(2).to_string(index=False))
print("\n--- noise labels (decision 3c; reporting only, never reverses a choice) ---")
print(pd.DataFrame(lab).round(2).to_string(index=False))
print(f"\nbootstrap rows saved: {len(pd.read_csv(PHASE5_BOOTSTRAP_PATH))} (expect 24); seed={SEED}, repeats={B}")

D39 re-check: defaults vs tuned  dMAE=+0.11  dMAPE=+0.01 pp  -> REVERSE: use library defaults
SUBMITTED model: HGB log_rpm -market_index +weekday
pre-run check: lr=0.1, max_iter=100, early_stopping=False, random_state=42

--- NO-WEEKDAY (submitted settings) ---
  wf-May  rows= 4913 ok=True dropped= 275 fit_rows= 18835/ 19110 MAE=157.1731
  wf-Jun  rows= 4783 ok=True dropped= 337 fit_rows= 23686/ 24023 MAE=210.7646
  wf-Jul  rows= 4912 ok=True dropped= 400 fit_rows= 28406/ 28806 MAE=122.3253
  wf-Aug  rows= 4759 ok=True dropped= 465 fit_rows= 33253/ 33718 MAE=93.1841

--- ERRORS KEPT (submitted settings) ---
  wf-May  rows= 4913 ok=True dropped=   0 fit_rows= 19110/ 19110 MAE=163.4220
  wf-Jun  rows= 4783 ok=True dropped=   0 fit_rows= 24023/ 24023 MAE=216.2601
  wf-Jul  rows= 4912 ok=True dropped=   0 fit_rows= 28806/ 28806 MAE=127.3362
  wf-Aug  rows= 4759 ok=True dropped=   0 fit_rows= 33718/ 33718 MAE=101.1178

(ref = SUBMITTED; wins_vs_ref = folds where the VARIANT beats it)
      